# 04 — Unsupervised anomaly detection

**Purpose.** Learn a model of *normal* traffic and score how unusual each flow is (Isolation Forest), choose a
threshold on the **validation** split, explore clustering (K-Means, DBSCAN) and run an unknown-attack hold-out
experiment (Proposal RQ4).

**Score convention (ML-05).** `anomaly_score = −IsolationForest.score_samples(x)`, which lies in (0, 1]:
**higher = more anomalous**; typical normal traffic scores ≈ 0.4–0.5. `is_anomaly = anomaly_score ≥ threshold`.
An anomaly is an *unusual* flow, **not necessarily an attack** — benign rare behaviour is flagged too, and attacks
that look like normal traffic are not.

| | |
|---|---|
| **Inputs** | processed splits + preprocessor (notebook 02) |
| **Outputs** | `staging/<family>/anomaly/isolation_forest.joblib`, `anomaly_meta.json`, experiment records |

In [ ]:
# --- Project bootstrap: make backend/app/ml and notebooks/nb_utils importable ---------------------
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not ((_root / "backend" / "app" / "ml").is_dir() and (_root / "notebooks").is_dir()):
    if _root.parent == _root:
        raise RuntimeError("Start Jupyter inside the CyberSentinel repository (for example from notebooks/).")
    _root = _root.parent
for _path in (str(_root / "backend"), str(_root / "notebooks")):
    if _path not in sys.path:
        sys.path.insert(0, _path)

from nb_utils import load_config

CFG = load_config()  # all settings come from CS_* environment variables (see nb_utils/config.py)
CFG.describe()

In [ ]:
import warnings

import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

from app.ml.constants import ANOMALY_SCORE_DIRECTION, BENIGN_LABEL
from app.ml.evaluation import metrics as M
from nb_utils import io, models, plotting
from nb_utils.experiments import record_experiment, save_json

warnings.filterwarnings("ignore")
STAGE = CFG.staging_dir / "anomaly"
STAGE.mkdir(parents=True, exist_ok=True)
FAST = CFG.fast_mode
TARGET_BENIGN_FPR = 0.01  # threshold = 99th percentile of benign validation scores

In [ ]:
splits_ = io.load_splits(CFG.processed_dir)
prep_report = io.load_json(CFG.processed_dir / "preprocessing_report.json")
preprocessor = io.load_object(CFG.processed_dir / "preprocessor.joblib")
FEATURES = [f["name"] for f in prep_report["input_features"]]
train, val = splits_["train"], splits_["validation"]
X_train, X_val = preprocessor.transform(train[FEATURES]), preprocessor.transform(val[FEATURES])
benign_train = (train["label_category"] == BENIGN_LABEL).to_numpy()
y_val, cat_val = val["label_binary"].to_numpy(), val["label_category"].to_numpy()


def anomaly_score(model, X):
    """CyberSentinel convention: higher = more anomalous."""
    return -model.score_samples(X)


print("train", X_train.shape, "(benign rows:", int(benign_train.sum()), ") | validation", X_val.shape)

## 1. Two training regimes
* **benign_only** — novelty detection: fit on benign training flows (uses labels only to select normal data).
* **all_unlabeled** — fully unsupervised: fit on all training flows without labels.

In [ ]:
variants = {
    "benign_only": models.fit_timed(models.make_isolation_forest(CFG.seed, CFG.n_jobs, FAST), X_train[benign_train]),
    "all_unlabeled": models.fit_timed(models.make_isolation_forest(CFG.seed, CFG.n_jobs, FAST), X_train),
}
{name: f"{seconds:.1f}s" for name, (_, seconds) in variants.items()}

## 2. Score direction check

In [ ]:
iso_benign = variants["benign_only"][0]
centre = np.median(X_train[benign_train], axis=0)
extreme = centre + 10 * X_train[benign_train].std(axis=0)
s_centre, s_extreme = anomaly_score(iso_benign, np.vstack([centre, extreme]))
print(f"score at benign centre = {s_centre:.3f}; score of an extreme synthetic point = {s_extreme:.3f}")
assert s_extreme > s_centre, "anomaly scores must increase for more unusual inputs"

## 3. Threshold selection on validation benign scores and validation metrics

In [ ]:
summaries, val_scores = {}, {}
for name, (model, seconds) in variants.items():
    s = anomaly_score(model, X_val)
    val_scores[name] = s
    benign_scores = s[y_val == 0]
    threshold = float(np.quantile(benign_scores, 1 - TARGET_BENIGN_FPR))
    median = float(np.median(benign_scores))
    upper = float(max(np.quantile(s, 0.999), threshold + 1e-6))
    if not median < threshold:  # degenerate score distributions (tiny samples)
        median = threshold - 1e-6
    metrics_val = M.anomaly_detection_metrics(y_val, s, threshold, cat_val)
    summaries[name] = {"threshold": threshold, "reference_points": {"benign_median": median, "threshold": threshold, "upper": upper},
                       "train_seconds": round(seconds, 3), "validation": metrics_val}
    at = metrics_val["at_threshold"]
    print(f"{name:14s} ROC-AUC={metrics_val['roc_auc']} PR-AUC={metrics_val['pr_auc']} "
          f"FPR@thr={at['false_positive_rate']} attack recall@thr={at['recall']}")

In [ ]:
CHOSEN = max(summaries, key=lambda k: summaries[k]["validation"]["pr_auc"] or -1.0)
chosen_scores = val_scores[CHOSEN]
plotting.score_histogram(chosen_scores, np.where(y_val == 1, "attack", "benign"), summaries[CHOSEN]["threshold"],
                         f"Isolation Forest ({CHOSEN}) — validation", CFG.figures_dir / "04_anomaly_histogram.png")
plotting.roc_pr_curves(y_val, chosen_scores, f"Isolation Forest ({CHOSEN})", CFG.figures_dir / "04_anomaly_roc_pr.png")
print("selected variant (validation PR-AUC):", CHOSEN)
pd.DataFrame(summaries[CHOSEN]["validation"]["flag_rate_by_category"]).T

In [ ]:
model, seconds = variants[CHOSEN]
io.save_object(model, STAGE / "isolation_forest.joblib")
anomaly_meta = {
    "training_population": CHOSEN,
    "score_definition": "-IsolationForest.score_samples(x); range (0, 1]; higher = more anomalous",
    "score_direction": ANOMALY_SCORE_DIRECTION,
    "threshold": summaries[CHOSEN]["threshold"],
    "threshold_method": f"{100 * (1 - TARGET_BENIGN_FPR):.0f}th percentile of benign validation scores (target benign FPR {TARGET_BENIGN_FPR:.0%})",
    "reference_points": summaries[CHOSEN]["reference_points"],
    "train_seconds": round(seconds, 3),
    "validation": summaries[CHOSEN]["validation"],
    "variants": summaries,
}
save_json(anomaly_meta, STAGE / "anomaly_meta.json")
print("staged", STAGE / "isolation_forest.joblib")

## 4. Clustering (exploratory only)
K-Means (k chosen by silhouette) and DBSCAN (eps from the k-distance curve) on a training sample. DBSCAN has no `predict` for unseen flows, so it is **not deployable**; both are used to understand structure only.

In [ ]:
sample = models.stratified_indices(train["label_category"].to_numpy(), 1500 if FAST else 20_000, CFG.seed)
X_s, cat_s = X_train[sample], train["label_category"].to_numpy()[sample]
ks = range(2, 7 if FAST else 13)
silhouettes = {k: float(silhouette_score(X_s, KMeans(n_clusters=k, n_init=5, random_state=CFG.seed).fit_predict(X_s),
                                         sample_size=min(5000, len(X_s)), random_state=CFG.seed)) for k in ks}
best_k = max(silhouettes, key=silhouettes.get)
kmeans_labels = KMeans(n_clusters=best_k, n_init=10, random_state=CFG.seed).fit_predict(X_s)
print("silhouette by k:", {k: round(v, 3) for k, v in silhouettes.items()}, "-> k =", best_k)
pd.crosstab(pd.Series(kmeans_labels, name="k-means cluster"), pd.Series(cat_s, name="category"))

In [ ]:
db_sample = X_s[: min(len(X_s), 1000 if FAST else 5000)]
neighbours = 10
eps = float(np.quantile(NearestNeighbors(n_neighbors=neighbours).fit(db_sample).kneighbors(db_sample)[0][:, -1], 0.9))
db_labels = DBSCAN(eps=eps, min_samples=neighbours).fit_predict(db_sample)
noise = db_labels == -1
print(f"DBSCAN eps={eps:.3f}: {len(set(db_labels)) - int(noise.any())} clusters, noise fraction {noise.mean():.1%}")
pd.crosstab(pd.Series(np.where(noise, "noise", "clustered"), name="DBSCAN"), pd.Series(cat_s[: len(db_sample)], name="category"))

## 5. Two-dimensional projection (PCA, validation sample)

In [ ]:
vis = models.stratified_indices(cat_val, 2000 if FAST else 10_000, CFG.seed)
pca = PCA(n_components=2, random_state=CFG.seed).fit(X_train[sample])
points = pca.transform(X_val[vis])
flags = np.where(chosen_scores[vis] >= anomaly_meta["threshold"], "flagged anomalous", "not flagged")
plotting.scatter_2d(points, cat_val[vis], "PCA of validation flows by category", CFG.figures_dir / "04_pca_category.png")
plotting.scatter_2d(points, flags, "PCA of validation flows by anomaly flag", CFG.figures_dir / "04_pca_anomaly.png")
print("explained variance ratio:", np.round(pca.explained_variance_ratio_, 3))

## 6. Unknown-attack hold-out experiment (RQ4)
For the most frequent attack categories: train a random forest **without** that category, then check on the validation rows of that category whether it is still caught — as another attack class, by the benign-only Isolation Forest (which never saw it), by either, or by neither.

In [ ]:
holdout = pd.DataFrame()
if CFG.run_holdout:
    iso, threshold = variants["benign_only"][0], summaries["benign_only"]["threshold"]
    labels_train = train["label_category"].to_numpy()
    counts = pd.Series(labels_train[labels_train != BENIGN_LABEL]).value_counts()
    held_out = [c for c in counts.index[:4] if (cat_val == c).sum() >= 5]
    rows = []
    for category in held_out:
        keep = labels_train != category
        idx = models.stratified_indices(labels_train[keep], 3000 if FAST else 100_000, CFG.seed)
        classes = sorted(set(labels_train[keep][idx]))
        rf = models.make_random_forest(CFG.seed, CFG.n_jobs, fast=True).fit(
            X_train[keep][idx], np.array([classes.index(c) for c in labels_train[keep][idx]]))
        target = cat_val == category
        as_attack = np.asarray(classes, dtype=object)[rf.predict(X_val[target])] != BENIGN_LABEL
        flagged = anomaly_score(iso, X_val[target]) >= threshold
        rows.append({"held_out_category": category, "validation_rows": int(target.sum()),
                     "classified_as_other_attack": float(as_attack.mean()), "flagged_anomalous": float(flagged.mean()),
                     "caught_by_either": float((as_attack | flagged).mean()), "missed_by_both": float((~as_attack & ~flagged).mean())})
    holdout = pd.DataFrame(rows)
    record_experiment(
        experiment_id=f"{CFG.family}-unknown-attack-holdout", experiment_type="unknown_attack_holdout",
        title="Unknown-attack hold-out (validation split)", family=CFG.family, rows=holdout,
        columns={"classified_as_other_attack": "share predicted as some other attack class",
                 "flagged_anomalous": "share flagged by the benign-only Isolation Forest",
                 "missed_by_both": "share neither classified as attack nor flagged"},
        config={"seed": CFG.seed, "fast_mode": FAST, "threshold": threshold},
        summary={"categories": held_out}, notes=["The held-out category is absent from the classifier's training data."],
        experiments_dir=CFG.experiments_dir, reports_dir=CFG.reports_dir)
holdout

## 7. Record the anomaly-detection experiment

In [ ]:
variant_rows = pd.DataFrame([
    {"variant": name, "roc_auc": s["validation"]["roc_auc"], "pr_auc": s["validation"]["pr_auc"],
     "threshold": s["threshold"], "false_positive_rate_at_threshold": s["validation"]["at_threshold"]["false_positive_rate"],
     "attack_recall_at_threshold": s["validation"]["at_threshold"]["recall"], "flag_rate": s["validation"]["flag_rate"],
     "train_seconds": s["train_seconds"]}
    for name, s in summaries.items()
])
record_experiment(
    experiment_id=f"{CFG.family}-anomaly-detection", experiment_type="anomaly_detection",
    title="Isolation Forest variants (validation split)", family=CFG.family, rows=variant_rows,
    columns={"roc_auc": "ROC-AUC of the anomaly score vs attack label", "pr_auc": "average precision",
             "false_positive_rate_at_threshold": "benign flows flagged at the selected threshold"},
    config={"seed": CFG.seed, "fast_mode": FAST, "target_benign_fpr": TARGET_BENIGN_FPR},
    summary={"selected_variant": CHOSEN, "flag_rate_by_category": summaries[CHOSEN]["validation"]["flag_rate_by_category"],
             "kmeans_best_k": best_k, "dbscan_noise_fraction": float(noise.mean())},
    notes=["Anomaly scores measure unusualness, not maliciousness.", "Threshold chosen on validation benign scores."],
    experiments_dir=CFG.experiments_dir, reports_dir=CFG.reports_dir)
variant_rows